# 07 — Validación frente al criterio experto

Compara las salidas de la aplicación con las de un evaluador experto sobre el mismo conjunto de hojas, tomando la etiqueta del conjunto de datos como verdad de campo.

Las dos columnas de la aplicación, `patogeno_app` y `severidad_app`, **se calculan en este notebook** ejecutando la cascada real `M_seg → M1 → M2 → severidad CIELab` con los mismos pesos, umbrales y preprocesamiento que el backend. El archivo de anotación aporta únicamente el juicio humano, que es el único dato no reproducible por código.

## Entradas

| Ruta | Contenido |
|---|---|
| `splits/severity_val/labels.csv` | `file_name`, `severidad_experto`, `patogeno_experto` |
| `splits/severity_val/images/<clase>/` | Hojas evaluadas, agrupadas por clase real |
| `outputs/model_seg.keras` | Segmentador entrenado en el notebook 02 |
| `outputs/model1_binary.keras` | Clasificador de estado sanitario, notebook 03 |
| `outputs/model2_pathogen.keras` | Clasificador de patógeno, notebook 04 |

## Salidas

| Ruta | Contenido |
|---|---|
| `outputs/validacion_experto_predicciones.csv` | Manifiesto por hoja: ruta, verdad de campo, experto y predicción de la app |
| `outputs/comparacion_app_vs_experto.json` | Métricas agregadas de severidad y de patógeno |
| `outputs/severidad_app_vs_experto.png` | Dispersión con línea de identidad y Bland-Altman |
| `outputs/patogeno_app_vs_experto.png` | Exactitud por clase, app frente a experto |
| `outputs/patogeno_confusion_app_experto.png` | Matrices de confusión de ambos |

Participó un solo evaluador y no se midió acuerdo inter-evaluador, por lo que el contraste es orientativo (véase la sección de limitaciones del artículo).

In [ ]:
!pip install -q pandas numpy matplotlib scipy scikit-learn opencv-python-headless

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score

SPLIT = Path('./splits')
OUT = Path('./outputs'); OUT.mkdir(exist_ok=True)
SEV_DIR = SPLIT / 'severity_val'
IMAGES_DIR = SEV_DIR / 'images'
DISEASED_GATE = 0.5

df = pd.read_csv(SEV_DIR / 'labels.csv')
df.columns = [c.strip().lower() for c in df.columns]
df['file_name'] = df['file_name'].astype(str).str.strip()

rutas, clase_real = {}, {}
for sub in sorted(IMAGES_DIR.iterdir()):
    if sub.is_dir():
        for f in sorted(sub.iterdir()):
            if f.is_file():
                rutas[f.name] = f
                clase_real[f.name] = sub.name

df['ruta'] = df['file_name'].map(rutas)
df['patogeno_real'] = df['file_name'].map(clase_real)

sin_imagen = df.loc[df['ruta'].isna(), 'file_name'].tolist()
assert not sin_imagen, f'Anotaciones sin imagen correspondiente: {sin_imagen}'

for col in ['patogeno_experto', 'patogeno_real']:
    df[col] = df[col].astype(str).str.strip().str.lower()
df['severidad_experto'] = pd.to_numeric(df['severidad_experto'], errors='coerce')
df = df.sort_values(['patogeno_real', 'file_name']).reset_index(drop=True)

print(f'Hojas evaluadas por el experto: {len(df)}')
print(df.groupby('patogeno_real').size().rename('hojas').to_string())

## Procedencia de las hojas evaluadas

Antes de calcular nada se comprueba de dónde salen las hojas que vio el experto. La comparación solo es informativa si proceden del conjunto de prueba independiente: si alguna estuviera en entrenamiento o validación, la aplicación la habría visto durante el ajuste y su acierto no sería comparable con el del evaluador.

In [ ]:
def _nombres(*directorios):
    encontrados = set()
    for d in directorios:
        if d.exists():
            encontrados.update(f.name for f in d.rglob('*') if f.is_file())
    return encontrados


en_test = _nombres(SPLIT / 'test' / 'clasificacion_binaria', SPLIT / 'test' / 'clasificacion_patogeno')
en_train = _nombres(SPLIT / 'train' / 'clasificacion_binaria', SPLIT / 'train' / 'clasificacion_patogeno')
en_val = _nombres(SPLIT / 'val' / 'clasificacion_binaria', SPLIT / 'val' / 'clasificacion_patogeno')
evaluadas = set(df['file_name'])

df['origen'] = df['file_name'].map(
    lambda n: 'test' if n in en_test else 'val' if n in en_val else 'train' if n in en_train else 'externo')

fuga = sorted(evaluadas & (en_train | en_val))
print(df['origen'].value_counts().rename('hojas').to_string())
print(f'\nCoincidencias con entrenamiento o validacion: {len(fuga)}')
if fuga:
    print('  ' + ', '.join(fuga[:10]) + (' ...' if len(fuga) > 10 else ''))
    print('  ADVERTENCIA: hay hojas vistas durante el ajuste; la comparacion no es independiente.')
else:
    print('  Ninguna: la comparacion es independiente del ajuste.')

## Inferencia: la cascada completa de la aplicación

Sobre cada hoja se reproduce el flujo del backend con los mismos pesos y umbrales que la aplicación desplegada:

1. **Normalización cromática** Shades-of-Gray, norma de Minkowski *p* = 6 y ganancia acotada a [0.6, 1.6].
2. **M_seg** delimita la hoja a 256 × 256 px; se conserva el mayor componente conexo.
3. **M1** (240 × 240, doble entrada) decide sana o enferma con el umbral desplegado (`diseased_gate = 0.5`).
4. **M2** (224 × 224, doble entrada) identifica el patógeno **solo si M1 clasifica la hoja como enferma**; en caso contrario la predicción es `sana`.
5. **Severidad CIELab** sobre la región segmentada, con los umbrales de `SeverityAnalyzer`, sujeta a la misma condición que M2.

Los pasos 4 y 5 se activan bajo la misma condición que en producción, de modo que las columnas comparadas con el experto son exactamente lo que la aplicación devolvería ante esas hojas. El manifiesto resultante deja por escrito, hoja a hoja, qué predijo cada parte de la cascada.

In [ ]:
import cv2
import tensorflow as tf
from PIL import Image

M_SEG = tf.keras.models.load_model(OUT / 'model_seg.keras', compile=False)
M1 = tf.keras.models.load_model(OUT / 'model1_binary.keras', compile=False)
M2 = tf.keras.models.load_model(OUT / 'model2_pathogen.keras', compile=False)


def _tam(modelo):
    alto, ancho = modelo.inputs[0].shape[1:3]
    return int(alto), int(ancho)


M1_SIZE, M2_SIZE = _tam(M1), _tam(M2)
_indices = json.load(open(OUT / 'class_indices_model2_pathogen.json', encoding='utf-8'))
CLASES_M2 = [nombre for nombre, _ in sorted(_indices.items(), key=lambda kv: kv[1])]
print(f'M1 {M1_SIZE} | M2 {M2_SIZE} -> {CLASES_M2}')


def chromatic_normalize(img_rgb):
    x = img_rgb.astype(np.float32)
    illum = np.power(np.mean(np.power(x, 6), axis=(0, 1)), 1.0 / 6.0)
    scale = np.clip(illum.mean() / (illum + 1e-6), 0.6, 1.6)
    return np.clip(x * scale, 0, 255).astype(np.uint8)


def leaf_mask(norm_256):
    prob = M_SEG.predict((norm_256.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0]
    mask = (np.argmax(prob, -1) == 1).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if count > 2:
        areas = stats[1:, cv2.CC_STAT_AREA]
        keep = 1 + int(np.argmax(areas))
        thr = 0.15 * float(areas.max())
        mask = np.isin(labels, [i + 1 for i, a in enumerate(areas) if a >= thr or i + 1 == keep]).astype(np.uint8)
    return mask


def enclosed_holes(leaf):
    padded = np.pad(leaf.astype(np.uint8), 1, constant_values=0)
    flood = padded.copy()
    cv2.floodFill(flood, np.zeros((padded.shape[0] + 2, padded.shape[1] + 2), np.uint8), (0, 0), 1)
    return (flood[1:-1, 1:-1] == 0) & ~leaf


def cielab_severity(norm_256, mask_256):
    lab = cv2.cvtColor(norm_256, cv2.COLOR_RGB2LAB)
    L, a, b = (lab[:, :, i].astype(np.int16) for i in range(3))
    leaf = mask_256.astype(bool)
    green = a < 123
    chlorosis = leaf & (~green) & (b > 150) & (L > 110)
    necrosis = leaf & (~green) & (L < 130) & (b > 130)
    soil = (np.abs(a - 128) < 14) & (np.abs(b - 128) < 20) & (L > 150)
    holes = enclosed_holes(leaf) & soil
    expected = int(np.count_nonzero(leaf)) + int(np.count_nonzero(holes)) or 1
    symptomatic = chlorosis | necrosis | holes
    return min(round(float(np.count_nonzero(symptomatic)) / expected * 100, 1), 100.0)


def _entradas(norm_256, mask_256, size):
    destino = (size[1], size[0])
    original = cv2.resize(norm_256, destino)
    aislada = cv2.resize(norm_256 * mask_256[:, :, np.newaxis], destino)
    return {'original': original[np.newaxis].astype(np.float32),
            'hoja_aislada': aislada[np.newaxis].astype(np.float32)}


def diagnosticar(ruta):
    img = np.array(Image.open(ruta).convert('RGB').resize((256, 256)))
    norm = chromatic_normalize(img)
    mask = leaf_mask(norm)
    p_enferma = 1.0 - float(M1.predict(_entradas(norm, mask, M1_SIZE), verbose=0)[0][0])
    if p_enferma < DISEASED_GATE:
        return 'sana', p_enferma, 0.0
    probs = M2.predict(_entradas(norm, mask, M2_SIZE), verbose=0)[0]
    return CLASES_M2[int(np.argmax(probs))], p_enferma, cielab_severity(norm, mask)


predicciones = [diagnosticar(ruta) for ruta in df['ruta']]
df['patogeno_app'] = [p[0] for p in predicciones]
df['prob_enferma'] = [round(p[1], 4) for p in predicciones]
df['severidad_app'] = [p[2] for p in predicciones]

MANIFIESTO = ['file_name', 'origen', 'patogeno_real', 'patogeno_experto', 'patogeno_app',
              'prob_enferma', 'severidad_experto', 'severidad_app']
df[MANIFIESTO].to_csv(OUT / 'validacion_experto_predicciones.csv', index=False)
print(f'Cascada ejecutada sobre {len(df)} hojas -> validacion_experto_predicciones.csv')
df[MANIFIESTO].head()

In [ ]:
sev = df.dropna(subset=['severidad_experto', 'severidad_app'])
exp_s = sev['severidad_experto'].to_numpy(float)
app_s = sev['severidad_app'].to_numpy(float)

r = float(np.corrcoef(exp_s, app_s)[0, 1]) if len(exp_s) > 1 else float('nan')
mx, my = exp_s.mean(), app_s.mean()
vx, vy = exp_s.var(), app_s.var()
cov = ((exp_s - mx) * (app_s - my)).mean()
ccc = float(2 * cov / (vx + vy + (mx - my) ** 2)) if (vx + vy) > 0 else float('nan')
mae = float(np.mean(np.abs(app_s - exp_s)))
rmse = float(np.sqrt(np.mean((app_s - exp_s) ** 2)))
diff = app_s - exp_s
bias = float(diff.mean()); sd = float(diff.std(ddof=1)) if len(diff) > 1 else 0.0
loa_lo, loa_hi = bias - 1.96 * sd, bias + 1.96 * sd
sev_metrics = {'n': int(len(exp_s)), 'pearson_r': round(r, 3), 'ccc': round(ccc, 3),
               'mae': round(mae, 2), 'rmse': round(rmse, 2), 'bias': round(bias, 2),
               'loa_low': round(loa_lo, 2), 'loa_high': round(loa_hi, 2),
               'within_10pct': round(float(np.mean(np.abs(diff) <= 10) * 100), 1),
               'within_20pct': round(float(np.mean(np.abs(diff) <= 20) * 100), 1)}
print('Severidad app vs experto:', json.dumps(sev_metrics, ensure_ascii=False))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.scatter(exp_s, app_s, alpha=0.7, color='steelblue')
a1.plot([0, 100], [0, 100], 'k--', lw=1, label='identidad')
a1.set_xlim(0, 100); a1.set_ylim(0, 100)
a1.set_xlabel('Severidad experto (%)'); a1.set_ylabel('Severidad app (%)')
a1.set_title(f'App vs experto (r={r:.3f}, CCC={ccc:.3f}, n={len(exp_s)})')
a1.legend(); a1.grid(alpha=0.3)
mean_pair = (exp_s + app_s) / 2
a2.scatter(mean_pair, diff, alpha=0.7, color='steelblue')
a2.axhline(bias, color='red', label=f'sesgo {bias:.1f}%')
a2.axhline(loa_hi, color='gray', ls='--', label=f'LoA 95% [{loa_lo:.1f}, {loa_hi:.1f}]')
a2.axhline(loa_lo, color='gray', ls='--')
a2.set_xlabel('Media app-experto (%)'); a2.set_ylabel('App - experto (%)')
a2.set_title('Bland-Altman'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / 'severidad_app_vs_experto.png', dpi=120); plt.show()

In [ ]:
real = df['patogeno_real'].to_numpy()
app = df['patogeno_app'].to_numpy()
expert = df['patogeno_experto'].to_numpy()

acc_app = float(np.mean(app == real))
acc_exp = float(np.mean(expert == real))
classes = sorted(set(real))
f1_app = float(f1_score(real, app, labels=classes, average='macro', zero_division=0))
f1_exp = float(f1_score(real, expert, labels=classes, average='macro', zero_division=0))

app_ok = app == real; exp_ok = expert == real
b = int(np.sum(exp_ok & ~app_ok))
c = int(np.sum(app_ok & ~exp_ok))
try:
    from scipy.stats import binomtest
    mcnemar_p = float(binomtest(min(b, c), b + c, 0.5).pvalue) if (b + c) > 0 else 1.0
except Exception:
    mcnemar_p = 1.0

pat_metrics = {'n': int(len(real)), 'accuracy_app': round(acc_app, 3), 'accuracy_experto': round(acc_exp, 3),
               'f1_macro_app': round(f1_app, 3), 'f1_macro_experto': round(f1_exp, 3),
               'discordantes_solo_experto': b, 'discordantes_solo_app': c, 'mcnemar_p': round(mcnemar_p, 4)}
print('Patogeno:', json.dumps(pat_metrics, ensure_ascii=False))
veredicto = 'APP >= EXPERTO' if acc_app >= acc_exp else 'EXPERTO > APP'
sig = 'significativa (p<0.05)' if mcnemar_p < 0.05 else 'no significativa'
print(f'  {veredicto} | diferencia {sig}')

json.dump({'severidad': sev_metrics, 'patogeno': pat_metrics},
          open(OUT / 'comparacion_app_vs_experto.json', 'w'), indent=2, ensure_ascii=False)

In [ ]:
acc_app_c, acc_exp_c = [], []
for cl in classes:
    m = real == cl
    acc_app_c.append(float(np.mean(app[m] == real[m])))
    acc_exp_c.append(float(np.mean(expert[m] == real[m])))
labels_plot = classes + ['GLOBAL']
acc_app_c.append(acc_app); acc_exp_c.append(acc_exp)

x = np.arange(len(labels_plot)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w / 2, acc_app_c, w, label=f'App (acc {acc_app:.2f})', color='seagreen')
b2 = ax.bar(x + w / 2, acc_exp_c, w, label=f'Experto (acc {acc_exp:.2f})', color='slategray')
ax.set_xticks(x); ax.set_xticklabels(labels_plot, rotation=20)
ax.set_ylim(0, 1.05); ax.set_ylabel('Exactitud vs patogeno real')
ax.set_title(f'Patogeno: App vs Experto vs Real (n={len(real)}, McNemar p={pat_metrics["mcnemar_p"]})')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bars in (b1, b2):
    for rect in bars:
        ax.annotate(f'{rect.get_height():.2f}', (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                    ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'patogeno_app_vs_experto.png', dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, title in [(axes[0], app, 'App'), (axes[1], expert, 'Experto')]:
    cm = confusion_matrix(real, pred, labels=classes)
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel('predicho'); ax.set_ylabel('real'); ax.set_title(f'{title} vs real')
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'patogeno_confusion_app_experto.png', dpi=120); plt.show()